In [ ]:
# --- Configuration ---
import os, sys, time, re, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
%matplotlib inline

sys.path.insert(0, str(Path.cwd().parent.parent))
from shared.data_loader import prepare_hierarchical_data_from_splits
from shared.metrics import (compute_all_metrics, compute_per_class_f1,
                             plot_confusion_matrix, plot_per_class_f1)
from shared.utils import set_seed

SEED = 42
N_JOBS = int(os.environ.get("N_JOBS", 26))  # Défaut 26, surchargeable via env
set_seed(SEED)

PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

print(f"Configuration terminée. N_JOBS={N_JOBS}")

In [ ]:
# --- Chargement des données ---
print("Chargement des splits train/val/test...")
t0 = time.time()

data = prepare_hierarchical_data_from_splits(
    train_path=str(DATASET_DIR / "train_split.jsonl"),
    val_path=str(DATASET_DIR / "val_split.jsonl"),
    test_path=str(DATASET_DIR / "test_split.jsonl"),
    n_samples=None,
    seed=SEED,
)

print(f"Chargement terminé en {time.time() - t0:.0f}s")
print(f"  Train : {len(data['X_train']):,} | Val : {len(data['X_val']):,} | Test : {len(data['X_test']):,}")
print(f"  L1 : {data['num_classes_l1']} | L2 : {data['num_classes_l2']}")


<a id="section-2"></a>
## 2. Entraînement Word2Vec (Skip-gram)

On entraîne un modèle Word2Vec Skip-gram sur le corpus d'entraînement.
Chaque mot est représenté par un vecteur dense de 200 dimensions.


In [ ]:
# --- Stopwords & Tokenisation ---
import nltk

try:
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))

all_stops = french_stops | english_stops

def tokenize(text):
    """Tokenise un texte : lowercase, suppression stopwords et tokens courts."""
    tokens = re.findall(r"\b\w+\b", str(text).lower())
    return [t for t in tokens if len(t) >= 3 and t not in all_stops]

print("Tokenisation du corpus d'entraînement...")
t0 = time.time()
train_tokens = [tokenize(text) for text in data["X_train"]]
print(f"Tokenisation terminée en {time.time() - t0:.0f}s")
print(f"  Documents : {len(train_tokens):,}")
print(f"  Tokens moyen/doc : {np.mean([len(t) for t in train_tokens]):.0f}")

In [ ]:
# --- Entraînement Word2Vec ---
from gensim.models import Word2Vec

print(f"Entraînement Word2Vec Skip-gram (200d, 10 epochs, {N_JOBS} workers)...")
t0 = time.time()

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=3,
    sg=1,  # Skip-gram
    workers=N_JOBS,
    epochs=10,
    seed=SEED,
)

w2v_time = time.time() - t0
print(f"Entraînement terminé en {w2v_time:.0f}s")
print(f"  Vocabulaire : {len(w2v_model.wv):,} mots")
print(f"  Dimensions : {w2v_model.wv.vector_size}")

<a id="section-3"></a>
## 3. Vectorisation des documents

Chaque document est représenté par la **moyenne des embeddings** de ses tokens.
Les tokens absents du vocabulaire Word2Vec sont ignorés.


In [ ]:
# --- Vectorisation par moyenne des embeddings ---
# Note : pas de joblib ici — le modèle Word2Vec est trop gros pour être
# sérialisé vers N workers (OOM avec 42 copies). La boucle Python est
# suffisamment rapide (~20 min pour 7.9M docs).
def doc_to_vec(tokens, model, vector_size):
    """Moyenne des embeddings des tokens présents dans le vocabulaire."""
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(vector_size)

def corpus_to_matrix(texts, w2v, vector_size):
    """Vectorise un corpus entier."""
    tokens_list = [tokenize(text) for text in texts]
    return np.array([doc_to_vec(t, w2v, vector_size) for t in tokens_list], dtype=np.float32)

print("Vectorisation des splits...")
t0 = time.time()
X_train_w2v = corpus_to_matrix(data["X_train"], w2v_model, 200)
X_val_w2v = corpus_to_matrix(data["X_val"], w2v_model, 200)
X_test_w2v = corpus_to_matrix(data["X_test"], w2v_model, 200)

print(f"Vectorisation terminée en {time.time() - t0:.0f}s")
print(f"  Train : {X_train_w2v.shape}")
print(f"  Val   : {X_val_w2v.shape}")
print(f"  Test  : {X_test_w2v.shape}")

<a id="section-4"></a>
## 4. Classification L1 (157 catégories)


In [ ]:
# --- Entraînement L1 ---
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# LinearSVC mono-thread : avec 157 classes et 200 features denses,
# c'est rapide (~5 min). OneVsRestClassifier OOM sur matrice dense 5.5M × 200.
print("Entraînement LinearSVC sur L1...")
t0 = time.time()

clf_l1 = LinearSVC(
    C=1.0, class_weight="balanced", max_iter=200, dual="auto", random_state=SEED,
)
clf_l1.fit(X_train_w2v, data["y_l1_train"])
train_time_l1 = time.time() - t0

preds_l1_val = clf_l1.predict(X_val_w2v)
preds_l1_test = clf_l1.predict(X_test_w2v)

f1_l1_val = f1_score(data["y_l1_val"], preds_l1_val, average="macro")
f1_l1_test = f1_score(data["y_l1_test"], preds_l1_test, average="macro")

print(f"L1 terminé en {train_time_l1:.0f}s")
print(f"  Val  F1 macro L1 : {f1_l1_val:.4f}")
print(f"  Test F1 macro L1 : {f1_l1_test:.4f}")

<a id="section-5"></a>
## 5. Classification L2 (6000+ sous-catégories)


In [ ]:
# --- Entraînement L2 ---
from sklearn.linear_model import SGDClassifier

# early_stopping désactivé : ~200 classes L2 n'ont qu'1 seul produit,
# incompatible avec StratifiedShuffleSplit
# n_jobs=1 : la matrice dense Word2Vec (4.4 Go) serait copiée intégralement
# vers chaque worker joblib → OOM avec n_jobs > 1
print("Entraînement SGDClassifier sur L2 (mono-thread, 200 features denses)...")
t0 = time.time()

clf_l2 = SGDClassifier(
    loss="hinge", class_weight="balanced", alpha=1e-4,
    max_iter=100, random_state=SEED, n_jobs=1,
)
clf_l2.fit(X_train_w2v, data["y_l2_train"])
train_time_l2 = time.time() - t0

preds_l2_val = clf_l2.predict(X_val_w2v)
preds_l2_test = clf_l2.predict(X_test_w2v)

f1_l2_val = f1_score(data["y_l2_val"], preds_l2_val, average="macro")
f1_l2_test = f1_score(data["y_l2_test"], preds_l2_test, average="macro")

print(f"L2 terminé en {train_time_l2:.0f}s")
print(f"  Val  F1 macro L2 : {f1_l2_val:.4f}")
print(f"  Test F1 macro L2 : {f1_l2_test:.4f}")

<a id="section-6"></a>
## 6. Métriques hiérarchiques


In [ ]:
# --- Métriques hiérarchiques ---
all_metrics = compute_all_metrics(
    labels_l1=data["y_l1_test"], preds_l1=preds_l1_test,
    labels_l2=data["y_l2_test"], preds_l2=preds_l2_test,
    l2_to_l1_mapping=data["l2_to_l1_mapping"],
)

print("=== Métriques Word2Vec + LinearSVC/SGD (Test Set) ===")
for k, v in all_metrics.items():
    if isinstance(v, float):
        print(f"  {k:30s} : {v:.4f}")

summary = pd.DataFrame([{
    "Modèle": "Word2Vec + LinearSVC/SGD",
    "F1 L1 macro": all_metrics.get("f1_l1_macro", f1_l1_test),
    "F1 L2 macro": all_metrics.get("f1_l2_macro", f1_l2_test),
    "HCR": all_metrics.get("hcr", 0),
}])
display(summary.style.format({
    "F1 L1 macro": "{:.4f}", "F1 L2 macro": "{:.4f}", "HCR": "{:.4f}",
}).set_caption("Word2Vec Baseline — dataset complet 5.5M"))

<a id="section-7"></a>
## 7. Visualisations


In [ ]:
# --- Matrices de confusion & F1 par classe ---
per_class_f1_l1 = compute_per_class_f1(data["y_l1_test"], preds_l1_test, data["l1_encoder"])

fig_cm = plot_confusion_matrix(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"],
    title="Matrice de confusion L1 — Word2Vec Baseline", top_n=30,
)
plt.show()

fig_f1 = plot_per_class_f1(per_class_f1_l1, title="F1 par classe L1 — Word2Vec", top_n=20)
plt.show()


In [ ]:
# --- F1 par classe L2 ---
per_class_f1_l2 = compute_per_class_f1(data["y_l2_test"], preds_l2_test, data["l2_encoder"])
fig_f1_l2 = plot_per_class_f1(per_class_f1_l2, title="F1 par classe L2 — Word2Vec", top_n=20)
plt.show()

f1_vals = list(per_class_f1_l2.values())
print(f"F1 L2 — Moyenne: {np.mean(f1_vals):.4f}, Médiane: {np.median(f1_vals):.4f}")
print(f"  Classes avec F1 = 0 : {sum(1 for v in f1_vals if v == 0)}")
print(f"  Classes avec F1 > 0.5 : {sum(1 for v in f1_vals if v > 0.5)}")


<a id="section-9"></a>
## 9. Conclusion

### Word2Vec vs TF-IDF
- Word2Vec capture des **relations sémantiques** (synonymes, analogies) que TF-IDF ignore
- Mais la moyenne des embeddings perd l'**information d'ordre** et de **fréquence** des termes
- TF-IDF reste généralement supérieur pour la classification de texte avec un classifieur linéaire

### Ce que HYDRA apporte
- **Embeddings contextuels** : CamemBERT produit des représentations dépendantes du contexte
- **Fine-tuning end-to-end** : l'encodeur s'adapte à la tâche de classification
- **Architecture hiérarchique** : modélisation explicite de la relation L1→L2

---
*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*
